# Phase 06 - Leaf segmentation (Method B)

Annotated field polygons, the leaf segmenter and the crop pipeline.
See `docs/instructions/06-segmentation-method-b.md`.

Cells whose first line is `#| export` are copied, in order, into the
generated `group3-starvenus.py` by `tools/export_py.py`. Keep outputs cleared before committing.

## Hand annotation of 30 field images (T6.1)

The segmenter of phase 06 is judged on **30 hand-annotated field images**, 6 per class. This notebook holds
the part the code owns: which images, where the annotators work, and the sanity figure. The polygons
themselves are drawn by a person (`docs/instructions/annotation-guide.md`) and converted by the dev-only tool
`tools/labelme_to_annotations.py` into `annotations/field_salient_leaf.json`, which `tools/export_py.py` then
embeds as the constant `FIELD_ANNOTATIONS`.

**Selection.** For every class, in `CLASS_NAMES` order: sort the field filenames, take the first 6 of a
permutation drawn from one `np.random.default_rng(2026)` stream. The 6 are split into **2 dev + 4 eval** with a
second seeded stream (`default_rng([2026, 1])`). Dev images may be used to tune composite realism and segmenter
settings; eval images are looked at once per final segmenter, for the reported IoU. The list is written to
`_work/annotations/selection.json` (committed) and the cached images are copied to
`_work/annotations/images/`, because the polygons must be drawn on the exact pixels the code sees (portrait,
long side 1280).

**Duplicates.** Two selected images (`bsf_5.jpg`, `rtf_12.jpg`) have a near-duplicate partner in the field
pool. `annotation_exclusions` returns the selected images *and* their duplicate-group partners; the composite
generator (`30_copy_paste.ipynb`) never uses any of them as a background.

In [ ]:
#| export
ANNOTATION_SEED = 2026
ANNOTATIONS_PER_CLASS = 6
ANNOTATION_DEV_PER_CLASS = 2
ANNOTATION_FORMAT = 1


def annotations_dir(data_root) -> Path:
    return work_dir(data_root) / "annotations"


def annotation_images_dir(data_root) -> Path:
    return annotations_dir(data_root) / "images"


def selection_path(data_root) -> Path:
    return annotations_dir(data_root) / "selection.json"


def annotation_selection(index: list[Record]) -> list[dict]:
    """The 30 field images to annotate (T6.1): ``ANNOTATIONS_PER_CLASS`` per class, as a list of
    ``{"filename", "cls", "set": "dev" | "eval"}`` in CLASS_NAMES order, then dev before eval, then filename.

    Deterministic: the candidates of a class (field images, sorted by filename) are permuted by one
    ``default_rng(2026)`` stream shared by the classes in CLASS_NAMES order, and ``ANNOTATION_DEV_PER_CLASS`` of the
    chosen images become dev through a second stream, ``default_rng([2026, 1])``."""
    rng = np.random.default_rng(ANNOTATION_SEED)
    dev_rng = np.random.default_rng([ANNOTATION_SEED, 1])
    entries = []
    for cls in CLASS_NAMES:
        candidates = sorted(r.filename for r in index if r.cls == cls and r.domain == "field")
        if len(candidates) < ANNOTATIONS_PER_CLASS:
            raise DatasetError(f"{cls}: {len(candidates)} field images, need {ANNOTATIONS_PER_CLASS} to annotate")
        picked = sorted(candidates[i] for i in rng.permutation(len(candidates))[:ANNOTATIONS_PER_CLASS])
        dev = {picked[i] for i in dev_rng.permutation(ANNOTATIONS_PER_CLASS)[:ANNOTATION_DEV_PER_CLASS]}
        rows = [{"filename": name, "cls": cls, "set": "dev" if name in dev else "eval"} for name in picked]
        entries.extend(sorted(rows, key=lambda row: (row["set"] != "dev", row["filename"])))
    return entries


def annotation_exclusions(selection: list[dict], groups) -> set[str]:
    """Filenames that must never be a composite background: the annotated images and every member of their
    near-duplicate groups (a duplicate would show the annotated scene to the segmenter with the wrong target)."""
    chosen = {row["filename"] for row in selection}
    excluded = set(chosen)
    for group in groups:
        if chosen & set(group):
            excluded.update(group)
    return excluded


def selection_document(data_root, index: list[Record]) -> dict:
    """The content of selection.json: the images with their cached path (relative to the data root) and size."""
    manifest = load_manifest(data_root)
    images = []
    for row in annotation_selection(index):
        entry = dict(row, image=f"{WORK_DIRNAME}/cache/{row['filename']}")
        size = manifest.get(row["filename"], {})
        if "width" in size:
            entry.update(width=size["width"], height=size["height"])
        images.append(entry)
    return {
        "format": ANNOTATION_FORMAT, "seed": ANNOTATION_SEED, "per_class": ANNOTATIONS_PER_CLASS,
        "dev_per_class": ANNOTATION_DEV_PER_CLASS, "images": images,
    }


def load_selection(data_root) -> list[dict]:
    path = selection_path(data_root)
    if not path.is_file():
        raise DatasetError(f"{path} does not exist; run the `prepare` command first")
    return json.loads(read_file_bytes(path).decode("utf-8"))["images"]


def write_annotation_selection(data_root, index: list[Record]) -> Path:
    """Write ``_work/annotations/selection.json`` (one image per line, so that the committed file diffs well)."""
    doc = selection_document(data_root, index)
    header = ",\n".join(f" {json.dumps(key)}: {json.dumps(value)}" for key, value in doc.items() if key != "images")
    images = ",\n".join("  " + json.dumps(entry) for entry in doc["images"])
    path = selection_path(data_root)
    write_text_atomic(path, "{\n" + header + ',\n "images": [\n' + images + "\n ]\n}\n")
    return path


def copy_annotation_images(data_root, selection: list[dict]) -> int:
    """Copy the cached images to ``_work/annotations/images/`` (annotate those, not the originals). Returns how many
    were copied; files that already match (same size) are left alone, so the step is idempotent."""
    target = annotation_images_dir(data_root)
    target.mkdir(parents=True, exist_ok=True)
    copied = 0
    for row in selection:
        source, destination = cache_dir(data_root) / row["filename"], target / row["filename"]
        if not source.is_file():
            raise DatasetError(f"{source} is missing; run `prepare` to build the cache")
        if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
            write_bytes_atomic(destination, read_file_bytes(source))
            copied += 1
    return copied

## Annotation file, validation and the overview figure

`FIELD_ANNOTATIONS` is `{filename: {"polygon": [[x, y], ...], "ambiguous": bool}}` with vertices normalised to
[0, 1] by the cached image size (x by width, y by height). It exists in the generated module only after
`annotations/field_salient_leaf.json` was committed and the export re-run; `embedded_field_annotations` returns
`None` before that. `validate_field_annotations` checks it against the selection (all 30 images and nothing else,
3 to 60 vertices inside the image, positive area).

`save_annotation_overview` draws `_work/figures/annotations_overview.png`: one row per class, the 2 dev images
first, then the 4 eval images, with the polygon outlined in yellow over a thin black halo (visible on green and
on brown) and a `?` marker for `ambiguous`. `prepare` regenerates it whenever the annotations exist, so a
wrong class, a polygon on the wrong leaf or an unfinished outline is visible at a glance.

In [ ]:
#| export
MAX_POLYGON_VERTICES = 60
ANNOTATION_OUTLINE = OKABE_ITO["yellow"]


def embedded_field_annotations():
    """The embedded ``FIELD_ANNOTATIONS`` constant, or None while annotations/field_salient_leaf.json does not exist."""
    return globals().get("FIELD_ANNOTATIONS")


def polygon_area(polygon) -> float:
    points = np.asarray(polygon, dtype=np.float64)
    x, y = points[:, 0], points[:, 1]
    return float(abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))) / 2.0)


def validate_field_annotations(annotations, selection: list[dict]) -> dict:
    """The annotations, checked against the selection: exactly the selected images, each with a polygon of 3 to
    ``MAX_POLYGON_VERTICES`` vertices inside [0, 1] x [0, 1] and positive area. Raises DatasetError."""
    wanted = [row["filename"] for row in selection]
    missing = sorted(set(wanted) - set(annotations))
    extra = sorted(set(annotations) - set(wanted))
    if missing or extra:
        raise DatasetError(f"annotations do not match the selection: missing {missing or 'none'}, unexpected {extra or 'none'}")
    for name in wanted:
        entry = annotations[name]
        polygon = entry.get("polygon") if isinstance(entry, dict) else None
        if not isinstance(polygon, (list, tuple)) or not 3 <= len(polygon) <= MAX_POLYGON_VERTICES:
            raise DatasetError(f"{name}: expected a polygon of 3 to {MAX_POLYGON_VERTICES} vertices")
        points = np.asarray(polygon, dtype=np.float64)
        if points.ndim != 2 or points.shape[1] != 2 or points.min() < 0.0 or points.max() > 1.0:
            raise DatasetError(f"{name}: polygon vertices must be [x, y] pairs normalised to [0, 1]")
        if polygon_area(points) <= 0.0:
            raise DatasetError(f"{name}: polygon has no area")
        if not isinstance(entry.get("ambiguous"), bool):
            raise DatasetError(f"{name}: 'ambiguous' must be true or false")
    return annotations


def save_annotation_overview(data_root, selection: list[dict], annotations: dict, path: Path | None = None) -> Path:
    """`annotations_overview.png`: the annotated images, a row per class (2 dev, then 4 eval), outlines on top."""
    plt = pyplot()
    path = path or figures_dir(data_root) / "annotations_overview.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = {cls: [r for r in selection if r["cls"] == cls] for cls in CLASS_NAMES}
    columns = max(len(v) for v in rows.values())
    with plt.rc_context({"font.size": 7}):
        fig = plt.figure(figsize=(5.8, 1.78 * len(CLASS_NAMES) + 0.25))
        grid = fig.add_gridspec(len(CLASS_NAMES), columns, left=0.055, right=0.998, top=0.965, bottom=0.014, wspace=0.04, hspace=0.12)
        for r, cls in enumerate(CLASS_NAMES):
            for c in range(columns):
                ax = fig.add_subplot(grid[r, c])
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
                if c >= len(rows[cls]):
                    continue
                row = rows[cls][c]
                rgb = read_cached_rgb(data_root, row["filename"])
                height, width = rgb.shape[:2]
                # Downsample once, with an area filter, to the pixels the tile occupies at FIGURE_DPI, so that matplotlib
                # draws the photo 1:1 (its own 8-bit resampler is skipped). Colours are never quantised.
                box = ax.get_position()
                tile = (max(1, round(box.width * fig.get_figwidth() * FIGURE_DPI)), max(1, round(box.height * fig.get_figheight() * FIGURE_DPI)))
                ax.imshow(cv2.resize(rgb, tile, interpolation=cv2.INTER_AREA), extent=(0, width, height, 0), interpolation="none")
                entry = annotations.get(row["filename"])
                if entry is not None:
                    points = np.asarray(entry["polygon"], dtype=np.float64) * (width, height)
                    closed = np.vstack([points, points[:1]])
                    ax.plot(closed[:, 0], closed[:, 1], color="black", linewidth=1.5, solid_joinstyle="round")
                    ax.plot(closed[:, 0], closed[:, 1], color=ANNOTATION_OUTLINE, linewidth=0.7, solid_joinstyle="round")
                    if entry.get("ambiguous"):
                        ax.text(0.04, 0.97, "?", transform=ax.transAxes, ha="left", va="top", fontsize=9, fontweight="bold",
                                color="white", bbox={"boxstyle": "round,pad=0.15", "fc": OKABE_ITO["vermilion"], "ec": "none"})
                ax.set_xlim(0, width)
                ax.set_ylim(height, 0)
                ax.set_xlabel(f"{row['filename']}", fontsize=5.5, labelpad=1.5)
                if r == 0:
                    ax.set_title(row["set"], fontsize=7, fontweight="bold", pad=2,
                                 color=OKABE_ITO["blue"] if row["set"] == "dev" else "black")
                if c == 0:
                    ax.set_ylabel(cls, fontsize=7.5, fontweight="bold", labelpad=3)
        fig.savefig(path, dpi=FIGURE_DPI)
        plt.close(fig)
    return path


def prepare_annotations(data_root, records: list[Record], out=print, *, annotations=None) -> dict:
    """T6.1 part of `prepare`: selection.json, the images to annotate, and (once annotated) the checked overview
    figure. Skipped with a message when a class has fewer than 6 field images (tiny test trees).

    ``annotations`` are the polygons to validate and draw. ``None`` means the embedded ``FIELD_ANNOTATIONS``, which
    describe the real selection and are therefore only used when the index is the full expected dataset (the same rule
    as the embedded duplicate groups); tests pass their own for synthetic trees."""
    field_counts = Counter(r.cls for r in records if r.domain == "field")
    if any(field_counts[c] < ANNOTATIONS_PER_CLASS for c in CLASS_NAMES):
        out(f"  annotate    skipped: fewer than {ANNOTATIONS_PER_CLASS} field images in some class")
        return {"selected": 0}
    write_annotation_selection(data_root, records)
    selection = load_selection(data_root)
    copied = copy_annotation_images(data_root, selection)
    dev = sum(row["set"] == "dev" for row in selection)
    out(f"  annotate    {len(selection)} field images selected ({dev} dev, {len(selection) - dev} eval), {copied} copied to {annotation_images_dir(data_root)}")
    figure = None
    full_dataset = class_domain_counts(records) == EXPECTED_COUNTS
    if annotations is None and full_dataset:
        annotations = embedded_field_annotations()
    if annotations is None:
        if full_dataset:
            out("              no FIELD_ANNOTATIONS yet (annotations/field_salient_leaf.json); see docs/instructions/annotation-guide.md")
        else:
            out("              embedded FIELD_ANNOTATIONS not checked: this is not the full dataset")
    else:
        validate_field_annotations(annotations, selection)
        figure = save_annotation_overview(data_root, selection, annotations)
        out(f"              {sum(a['ambiguous'] for a in annotations.values())} ambiguous; overview figure {figure.name}")
    return {"selected": len(selection), "copied": copied, "figure": figure}

## Segmenter (T6.2)

The segmenter learns to find *the leaf the photographer aimed at* **without a single real field mask**. Its targets are
only the two kinds of image for which a mask exists: composites (the pasted leaf is the target, everything else, real
field leaves included, is background) and white-background images with an accepted pseudo-mask.

* **Model.** `torchvision.models.segmentation.lraspp_mobilenet_v3_large` with the `COCO_WITH_VOC_LABELS_V1` weights; both
  classifier convolutions of its LRASPP head are replaced by 1-output-channel ones (one leaf logit per pixel). It has 3.2 M
  parameters. The one architecture swap that 06 allows, `deeplabv3_mobilenet_v3_large`, is selectable through the
  `backbone` field of the `S-seg` registry row; it is decided on the dev annotations only.
* **Training data of split `k`** (no val or test pixel): per epoch 1000 samples, 50% composites made on the fly by
  `CompositeSource` from the white **train** leaves on same-class field **train** backgrounds (never an annotated image),
  and 50% white **train** images with their accepted pseudo-masks (`WHITE_REJECTS` excluded), downscaled to the
  composites' resolution (half the cache) so that sharpness does not tell the two kinds apart. Image and mask go through
  the augmentation of T3.2 *together*: they are stacked into one 4-channel array, so the random crop (60-100% of the area),
  the flips, the +-15 degree rotation (reflected border) and the resize touch both identically; colour jitter touches the
  image only. Network input is 512 x 256 (H x W), portrait. Every random draw is a function of `(seed, epoch, index)`.
* **Loss and optimiser.** BCE + soft Dice (Dice per image, then averaged); AdamW, lr 3e-4, weight decay 1e-4, 40 epochs of 1000
  samples, batch 16, cosine decay to 0 stepped per optimiser step, AMP on CUDA only.
* **Checkpoint selection.** After each epoch the IoU (threshold 0.5 on the raw sigmoid, mean over images) is measured on
  (a) 200 *validation composites* (white **val** leaves on field **val** backgrounds of the same class, fixed seed) and
  (b) every white **val** image with an accepted pseudo-mask. The score is the mean of the two; the best epoch is kept. When a
  split has no composite pair in val (tiny smoke data) the score is the white IoU alone.

In [ ]:
#| export
SEG_ARCHITECTURES = {
    "lraspp_mobilenet_v3_large": SEGMENTER_WEIGHTS,
    "deeplabv3_mobilenet_v3_large": "DeepLabV3_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1",
}
SEG_INPUT_SIZE = (512, 256)  # (height, width) of the segmenter input
SEG_SAMPLES_PER_EPOCH = 1000
SEG_COMPOSITE_SHARE = 0.5  # share of composites among the training samples of an epoch
SEG_EPOCHS, SEG_BATCH_SIZE, SEG_LR, SEG_WEIGHT_DECAY = 40, 16, 3e-4, 1e-4
SEG_VAL_COMPOSITES = 200  # validation composites for the checkpoint choice
SEG_VAL_SEED = 2027  # fixed seed of the validation composites
SEG_THRESHOLD = 0.5  # sigmoid threshold of the mask
SEG_MIN_AREA = 0.02  # a component must cover at least this fraction of the image
SEG_MARGIN = 0.15  # the crop box is the mask's bounding box plus this fraction of its side on every side
SEG_JITTER = 0.05  # Method B training: scale and shift of the crop box by +-5%
# White training samples are augmented at the resolution the composites are made at (half the cache): the two kinds of
# sample then look alike in sharpness, and the augmentation costs a quarter (the network input is 512x256 anyway).
SEG_WHITE_SCALE = COMPOSITE_TRAIN_SCALE
SEG_CONSTANTS = {
    "input": SEG_INPUT_SIZE, "samples": SEG_SAMPLES_PER_EPOCH, "composite_share": SEG_COMPOSITE_SHARE,
    "val_composites": SEG_VAL_COMPOSITES, "val_seed": SEG_VAL_SEED, "threshold": SEG_THRESHOLD, "white_scale": SEG_WHITE_SCALE,
    "version": 1,
}
SEG_RECIPE = hashlib.sha1(json.dumps(SEG_CONSTANTS, sort_keys=True).encode()).hexdigest()[:12]
BOX_CONSTANTS = {"threshold": SEG_THRESHOLD, "min_area": SEG_MIN_AREA, "margin": SEG_MARGIN, "jitter": SEG_JITTER, "version": 1}
BOX_RECIPE = hashlib.sha1(json.dumps(BOX_CONSTANTS, sort_keys=True).encode()).hexdigest()[:12]
SEG_HISTORY_COLUMNS = ["epoch", "lr", "train_loss", "val_iou_composite", "val_iou_white", "val_score", "best", "seconds"]


def seg_size(size_div: int = 1) -> tuple[int, int]:
    """(height, width) of the segmenter input; ``size_div`` > 1 is for smoke tests only."""
    return max(32, SEG_INPUT_SIZE[0] // size_div), max(16, SEG_INPUT_SIZE[1] // size_div)


def build_segmenter(arch: str = "lraspp_mobilenet_v3_large", *, pretrained: bool = True, data_root=None) -> nn.Module:
    """The leaf segmenter: a torchvision segmentation model whose classifier has ONE output channel (the leaf logit).
    ``pretrained=True`` starts from the COCO_WITH_VOC_LABELS_V1 weights (read from, and downloaded to,
    ``<data_root>/_work/weights``) and replaces the 21-class heads; ``False`` builds the same architecture with random
    weights (tests, predict, which load a checkpoint)."""
    if arch not in SEG_ARCHITECTURES:
        raise ValueError(f"unknown segmenter {arch!r}, expected one of {tuple(SEG_ARCHITECTURES)}")
    if data_root is not None and pretrained:
        set_torch_home(data_root)
    builder = getattr(torchvision.models.segmentation, arch)
    if not pretrained:
        return builder(weights=None, weights_backbone=None, num_classes=1)
    model = builder(weights=torchvision.models.get_weight(SEG_ARCHITECTURES[arch]))
    if arch.startswith("lraspp"):
        head = model.classifier
        head.low_classifier = nn.Conv2d(head.low_classifier.in_channels, 1, 1)
        head.high_classifier = nn.Conv2d(head.high_classifier.in_channels, 1, 1)
    else:
        model.classifier[-1] = nn.Conv2d(model.classifier[-1].in_channels, 1, 1)
        model.aux_classifier = None
    return model


def seg_logits(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    """Leaf logits [N, H, W] for a normalised batch [N, 3, H, W]."""
    return model(x)["out"][:, 0]


def seg_augment(rgb: np.ndarray, mask: np.ndarray, rng: np.random.Generator, size: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    """The T3.2 training augmentation applied to an image and its mask together: both are stacked into one 4-channel
    array, so crop, flips, rotation and resize act identically; colour jitter acts on the image only. Returns the image
    (uint8 [H, W, 3]) and the mask (uint8 0/1 [H, W]) at ``size``."""
    stacked = np.dstack([rgb, mask])
    stacked = random_rotation(random_flips(random_crop(stacked, rng), rng), rng)
    stacked = resize_exact(np.ascontiguousarray(stacked), *size)
    return color_jitter(np.ascontiguousarray(stacked[..., :3]), rng), (stacked[..., 3] >= 128).astype(np.uint8)


def seg_eval_arrays(rgb: np.ndarray, mask: np.ndarray, size: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    """The evaluation transform: resize only. Returns (uint8 image, uint8 0/1 mask)."""
    return resize_exact(rgb, *size), (resize_exact(mask, *size) >= 128).astype(np.uint8)


class SegmenterDataset:
    """The training samples of one segmenter epoch: ``n_samples`` of them, ``composite_share`` composites (from
    ``composites``, a `CompositeSource`) and the rest white train images with accepted pseudo-masks, in this order
    (indices 0..n_composites-1 are composites). ``dataset[(i, epoch)]`` returns ``(uint8 CHW image, uint8 1HW mask, i)``;
    the white image used for slot ``j`` of an epoch and every random draw are functions of ``(seed, epoch, i)`` only."""

    def __init__(self, data_root, whites: list[Record], composites, *, size, n_samples=SEG_SAMPLES_PER_EPOCH, seed=0,
                 composite_share=SEG_COMPOSITE_SHARE):
        self.data_root, self.whites, self.composites = Path(data_root), list(whites), composites
        self.size, self.seed, self.epoch = tuple(size), _sample_seed(seed), 0
        n_composites = 0 if composites is None else n_samples if not self.whites else round(n_samples * composite_share)
        self.n_composites = n_composites if composites is None else min(n_composites, len(composites))
        self.n_samples = self.n_composites + (n_samples - self.n_composites if self.whites else 0)
        if self.n_samples == 0:
            raise ExperimentError("the segmenter has no training data (no white train image with a mask, no composites)")

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, key):
        index, epoch = key if isinstance(key, tuple) else (key, self.epoch)
        if not 0 <= index < self.n_samples:
            raise IndexError(index)
        if index < self.n_composites:
            rgb, mask, _, _ = self.composites.load_with_mask(index, epoch)
        else:
            order = np.random.default_rng([self.seed, int(epoch), 91]).permutation(len(self.whites))
            record = self.whites[int(order[(index - self.n_composites) % len(order)])]
            rgb, mask = load_leaf(self.data_root, record, SEG_WHITE_SCALE)
        rng = np.random.default_rng([self.seed, int(epoch), int(index)])
        image, target = seg_augment(rgb, mask, rng, self.size)
        return torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))), torch.from_numpy(target[None]), index


@dataclass
class SegEvalSet:
    """Fixed validation images with their masks, already at network size."""

    images: torch.Tensor  # [N, 3, H, W] uint8
    masks: torch.Tensor  # [N, H, W] bool
    names: list[str]

    def __len__(self) -> int:
        return len(self.names)


def seg_eval_set(items: list[tuple]) -> SegEvalSet:
    """``items`` = [(name, uint8 RGB at network size, uint8 0/1 mask at network size)]."""
    if not items:
        return SegEvalSet(torch.empty(0, 3, 0, 0, dtype=torch.uint8), torch.empty(0, 0, 0, dtype=torch.bool), [])
    images = torch.from_numpy(np.stack([i.transpose(2, 0, 1) for _, i, _ in items]))
    masks = torch.from_numpy(np.stack([m for _, _, m in items]).astype(bool))
    return SegEvalSet(images, masks, [n for n, _, _ in items])


def white_mask_records(data_root, records: list[Record], filenames, rejects=None) -> list[Record]:
    """White images among ``filenames`` (a split part) that have an accepted pseudo-mask (not rejected, see `load_white_rejects`), in split order."""
    rejects = load_white_rejects(data_root, records) if rejects is None else rejects
    return [r for r in select_records(records, filenames, ("white",)) if r.filename not in rejects]


def build_seg_val_sets(data_root, split_seed: int, records: list[Record], size, *, n_composites: int = SEG_VAL_COMPOSITES,
                       limit: int | None = None) -> tuple[SegEvalSet, SegEvalSet]:
    """(validation composites, white validation images) of a split, at network ``size``. The composites put white **val**
    leaves on field **val** backgrounds of the same class (classes in turn, fixed seed `SEG_VAL_SEED`); none exist when the
    split has no such pair. ``limit`` (--fast) caps both sets."""
    split = load_split(data_root, split_seed)
    white = white_mask_records(data_root, records, split["val"])
    build_white_masks(data_root, white)
    if limit:
        white = subsample_records(white, limit)
    white_items = [(r.filename, *seg_eval_arrays(read_cached_rgb(data_root, r.filename), read_white_mask(data_root, r.filename), size)) for r in white]
    pool = CompositePool(data_root, split_seed, records, part="val")
    classes = sorted({r.cls_idx for r in pool.leaves})
    composite_items = []
    rng = np.random.default_rng(SEG_VAL_SEED)
    for i in range((min(n_composites, limit) if limit else n_composites) if classes else 0):
        leaf, background = pool.sample(rng, classes[i % len(classes)])
        image, mask, _ = make_composite(leaf, background, rng, data_root=data_root, scale=COMPOSITE_TRAIN_SCALE)
        composite_items.append((f"composite:{leaf.filename}+{background.filename}", *seg_eval_arrays(image, mask, size)))
    return seg_eval_set(composite_items), seg_eval_set(white_items)

In [ ]:
#| export
def seg_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """BCE on the logits plus soft Dice per image (averaged). ``logits`` [N, H, W], ``target`` [N, H, W] in {0, 1}."""
    logits, target = logits.float(), target.float()
    bce = F.binary_cross_entropy_with_logits(logits, target)
    probs = torch.sigmoid(logits)
    inter = (probs * target).sum(dim=(1, 2))
    dice = 1.0 - (2.0 * inter + 1.0) / (probs.sum(dim=(1, 2)) + target.sum(dim=(1, 2)) + 1.0)
    return bce + dice.mean()


def mask_iou(pred: np.ndarray, truth: np.ndarray) -> np.ndarray:
    """Per-image IoU of boolean masks [N, H, W] (1.0 where both are empty)."""
    inter = (pred & truth).sum(axis=(1, 2)).astype(np.float64)
    union = (pred | truth).sum(axis=(1, 2)).astype(np.float64)
    return np.where(union > 0, inter / np.maximum(union, 1.0), 1.0)


@torch.no_grad()
def segment_probabilities(model: nn.Module, images: torch.Tensor, device, *, batch_size: int = 32, amp: bool = False) -> np.ndarray:
    """Leaf probabilities [N, H, W] (float32, numpy) for a uint8 image tensor [N, 3, H, W]."""
    model.eval()
    out = []
    for start in range(0, len(images), batch_size):
        x = normalize_batch(images[start : start + batch_size], device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            logits = seg_logits(model, x)
        out.append(torch.sigmoid(logits.float()).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, 0, 0), dtype=np.float32)


def seg_val_iou(model: nn.Module, eval_set: SegEvalSet, device, *, amp: bool = False) -> float | None:
    """Mean per-image IoU of the thresholded probabilities on a SegEvalSet (None for an empty set)."""
    if not len(eval_set):
        return None
    pred = segment_probabilities(model, eval_set.images, device, amp=amp) > SEG_THRESHOLD
    return float(mask_iou(pred, eval_set.masks.numpy()).mean())


def seg_score(composite_iou, white_iou) -> float:
    """Checkpoint score: the mean of the validation-composite IoU and the white-val IoU (whichever exist)."""
    values = [v for v in (composite_iou, white_iou) if v is not None]
    return float(np.mean(values)) if values else float("nan")


def train_segmenter(cfg: ExperimentConfig, model: nn.Module, train_ds: SegmenterDataset, val_composites: SegEvalSet,
                    val_white: SegEvalSet, device: str, *, seed: int, num_workers: int = 0, mp_context=None, log=print) -> TrainOutcome:
    """AdamW + cosine (per optimiser step, no warmup), BCE + soft Dice, AMP on CUDA. Returns the epoch with the best
    validation score (`seg_score`); every epoch is evaluated, nothing stops early."""
    started = time.time()
    amp = device == "cuda"
    sampler = EpochSampler(len(train_ds), seed)
    drop_last = len(train_ds) % cfg.batch_size == 1  # a last batch of one sample breaks BatchNorm
    loader = make_loader(train_ds, cfg.batch_size, sampler=sampler, num_workers=num_workers, drop_last=drop_last,
                         pin_memory=device == "cuda", mp_context=mp_context)
    total_steps = cfg.epochs * len(loader)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    best_state, best_epoch, best_score, history, step = None, 0, -math.inf, [], 0
    for epoch in range(1, cfg.epochs + 1):
        epoch_started = time.time()
        sampler.set_epoch(epoch - 1)
        train_ds.epoch = epoch - 1
        model.train()
        loss_sum, seen, lr_now = 0.0, 0, cfg.lr
        for images, masks, _ in loader:
            lr_now = lr_at_step(step, base_lr=cfg.lr, warmup_steps=0, total_steps=total_steps)
            for group in optimizer.param_groups:
                group["lr"] = lr_now
            x, y = normalize_batch(images, device), masks[:, 0].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                loss = seg_loss(seg_logits(model, x), y)
            loss_value = loss.item()
            if not math.isfinite(loss_value):
                raise ExperimentError(f"{cfg.exp_id}: the training loss became {loss_value} (diverged)")
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss_value * len(y)
            seen += len(y)
            step += 1
        composite_iou, white_iou = seg_val_iou(model, val_composites, device, amp=amp), seg_val_iou(model, val_white, device, amp=amp)
        score = seg_score(composite_iou, white_iou)
        improved = best_state is None or score > best_score
        if improved:
            best_state, best_epoch, best_score = cpu_state_dict(model), epoch, score
        history.append({"epoch": epoch, "lr": lr_now, "train_loss": loss_sum / max(1, seen), "val_iou_composite": composite_iou,
                        "val_iou_white": white_iou, "val_score": score, "best": int(improved), "seconds": time.time() - epoch_started})
        fmt = lambda v: "n/a" if v is None else f"{v:.3f}"  # noqa: E731
        log(f"  epoch {epoch:2d}/{cfg.epochs}  lr {lr_now:.2e}  loss {history[-1]['train_loss']:.3f}  val IoU composites "
            f"{fmt(composite_iou)} white {fmt(white_iou)}  score {score:.3f}{' *' if improved else '  '}  {history[-1]['seconds']:.1f}s")
    return TrainOutcome(best_state, best_epoch, best_score, history, False, time.time() - started)

## From mask to crop box (post-processing) and the boxes of Method B (T6.3)

`postprocess_probability` turns a probability map into a crop box exactly as 06 prescribes:

1. threshold at 0.5 and take the 8-connected components;
2. among the components that cover **at least 2% of the image**, keep the one with the largest `area x centrality`, where
   centrality is `1 - (distance of the centroid from the image centre) / (half diagonal)`, so a large leaf away from the
   centre can lose against a slightly smaller one in the middle;
3. if no component reaches 2%: **fallback**, the crop box is the full image (the case is counted, per domain, in
   `boxes.json` and `seg_metrics.json`; its mask is empty, so it scores IoU 0);
4. the box is the component's bounding box plus 15% of its width / height on every side. At crop time (`crop_box_pixels`)
   it is grown along its relatively short side to the classifier's input aspect (448x224: height = 2 x width; 320x320:
   square), shifted inside the image when it sticks out (this keeps the aspect and the context), and clamped where the image
   itself is smaller than the box. Context is kept on purpose (D9): a slightly wrong mask still gives a usable crop.

**`boxes.json`** (`_work/runs/S-seg/split_<k>/`) holds, for **every cached image** (white and field alike), the
segmenter-`k` box as fractions of the image (`tight` = the bounding box, `box` = with the 15% margin, unclamped, and the
`fallback` flag). The aspect step is *not* baked in: it depends on the classifier's input mode and is applied by the hook, so
the file stays valid whichever input mode gate G0 chooses. Its sha256 is recorded in the metrics of the E4-B runs that use it.

**The Method B hook.** `SegCropHook(boxes, aspect, jitter)` is the `input_hook` of T3.2: it crops the image to the box
(fallback: the full image). At train time (`jitter = SEG_JITTER`) the box is scaled by a factor in [0.95, 1.05] and shifted by
up to +-5% of its width and height, drawn from the sample's own random stream; at evaluation time it is used exactly.
`LiveCropHook` does the same with a loaded segmenter instead of stored boxes: it is what `predict` uses for an image that
has no stored box (segmenter, then crop, then classifier).

In [ ]:
#| export
@dataclass(frozen=True)
class SegPost:
    """The result of post-processing one probability map (all coordinates are fractions of the image)."""

    mask: np.ndarray  # bool [H, W]: the selected component (all False on fallback)
    tight: tuple | None  # (x0, y0, x1, y1) bounding box of the component; None on fallback
    box: tuple  # tight box plus the margin, not clamped; (0, 0, 1, 1) on fallback
    fallback: bool
    area: float  # fraction of the image covered by the selected component (0 on fallback)


def postprocess_probability(prob: np.ndarray, *, threshold: float = SEG_THRESHOLD, min_area: float = SEG_MIN_AREA,
                            margin: float = SEG_MARGIN) -> SegPost:
    """Mask -> crop box (steps 1-4 of the section above, except the aspect step). ``prob`` is [H, W] in [0, 1]."""
    height, width = prob.shape
    count, labels, stats, centroids = cv2.connectedComponentsWithStats((prob > threshold).astype(np.uint8), connectivity=8)
    centre = np.array([(width - 1) / 2.0, (height - 1) / 2.0])
    half_diagonal = float(np.hypot(width - 1, height - 1)) / 2.0 or 1.0
    best, best_score = 0, -1.0
    for i in range(1, count):
        area = stats[i, cv2.CC_STAT_AREA] / float(height * width)
        if area < min_area:
            continue
        centrality = 1.0 - float(np.linalg.norm(centroids[i] - centre)) / half_diagonal
        if area * centrality > best_score:
            best, best_score = i, area * centrality
    if best == 0:
        return SegPost(np.zeros((height, width), bool), None, (0.0, 0.0, 1.0, 1.0), True, 0.0)
    x, y, w, h = (int(stats[best, c]) for c in (cv2.CC_STAT_LEFT, cv2.CC_STAT_TOP, cv2.CC_STAT_WIDTH, cv2.CC_STAT_HEIGHT))
    x0, y0, x1, y1 = x / width, y / height, (x + w) / width, (y + h) / height
    box = (x0 - margin * (x1 - x0), y0 - margin * (y1 - y0), x1 + margin * (x1 - x0), y1 + margin * (y1 - y0))
    return SegPost(labels == best, (x0, y0, x1, y1), box, False, float(stats[best, cv2.CC_STAT_AREA]) / (height * width))


def classifier_aspect(input_mode: str, size_div: int = 1) -> float:
    """Height / width of the classifier input (2.0 for rect, 1.0 for squash and pad)."""
    height, width = input_size(input_mode, size_div)
    return height / width


def fit_box(cx: float, cy: float, bw: float, bh: float, width: int, height: int) -> tuple[int, int, int, int]:
    """Pixel box of size (bw, bh) around (cx, cy): clamped to the image size, shifted inside the image, rounded outward to
    whole pixels (at least 1 x 1)."""
    bw, bh = min(bw, float(width)), min(bh, float(height))
    x0 = min(max(cx - bw / 2.0, 0.0), width - bw)
    y0 = min(max(cy - bh / 2.0, 0.0), height - bh)
    ix0, iy0 = int(math.floor(x0 + 1e-6)), int(math.floor(y0 + 1e-6))
    ix1, iy1 = min(width, max(ix0 + 1, int(math.ceil(x0 + bw - 1e-6)))), min(height, max(iy0 + 1, int(math.ceil(y0 + bh - 1e-6))))
    return ix0, iy0, ix1, iy1


def crop_box_pixels(box, aspect: float, width: int, height: int) -> tuple[int, int, int, int]:
    """The crop of an image of ``width`` x ``height`` for a stored (fractional, margin) ``box``: grown along its relatively
    short side to height / width = ``aspect``, then shifted inside the image and clamped (see the section text)."""
    x0, y0, x1, y1 = box[0] * width, box[1] * height, box[2] * width, box[3] * height
    bw, bh = max(x1 - x0, 1.0), max(y1 - y0, 1.0)
    if bh < bw * aspect:
        bh = bw * aspect
    else:
        bw = bh / aspect
    return fit_box((x0 + x1) / 2.0, (y0 + y1) / 2.0, bw, bh, width, height)


def jitter_crop(crop: tuple[int, int, int, int], rng: np.random.Generator, width: int, height: int, amount: float = SEG_JITTER):
    """Method B training augmentation of a pixel crop: scaled by a factor in 1 +- ``amount`` (both sides alike, so the aspect
    is kept) and shifted by up to ``amount`` of its width / height, then fitted into the image again."""
    x0, y0, x1, y1 = crop
    scale, dx, dy = rng.uniform(1 - amount, 1 + amount), rng.uniform(-amount, amount), rng.uniform(-amount, amount)
    bw, bh = x1 - x0, y1 - y0
    return fit_box((x0 + x1) / 2.0 + dx * bw, (y0 + y1) / 2.0 + dy * bh, bw * scale, bh * scale, width, height)


class SegCropHook:
    """`input_hook` of Method B: crop the image to the stored box of its filename (see the section text). ``jitter`` > 0
    makes the hook stochastic: the dataset then passes it a random generator derived from (seed, epoch, index)."""

    def __init__(self, boxes: dict, aspect: float, jitter: float = 0.0):
        self.boxes, self.aspect, self.jitter = boxes, float(aspect), float(jitter)
        self.stochastic = self.jitter > 0

    def __call__(self, rgb: np.ndarray, record, rng: np.random.Generator | None = None) -> np.ndarray:
        if record is None:
            raise ExperimentError("the stored-box crop needs the image's record (use LiveCropHook for ad-hoc images)")
        entry = self.boxes.get(record.filename)
        if entry is None:
            raise ExperimentError(f"boxes.json has no box for {record.filename}; rebuild it with `run --exp S-seg`")
        height, width = rgb.shape[:2]
        crop = (0, 0, width, height) if entry["fallback"] else crop_box_pixels(entry["box"], self.aspect, width, height)
        if self.jitter > 0 and rng is not None:
            crop = jitter_crop(crop, rng, width, height, self.jitter)
        x0, y0, x1, y1 = crop
        return np.ascontiguousarray(rgb[y0:y1, x0:x1])


@torch.no_grad()
def segmenter_posts(model: nn.Module, rgbs: list[np.ndarray], device, size, *, amp: bool = False) -> list[SegPost]:
    """Post-processed segmentation of full-size RGB images (each is resized to the network ``size`` first)."""
    batch = torch.from_numpy(np.stack([resize_exact(rgb, *size).transpose(2, 0, 1) for rgb in rgbs]))
    return [postprocess_probability(p) for p in segment_probabilities(model, batch, device, batch_size=len(rgbs), amp=amp)]


class LiveCropHook:
    """`input_hook` that segments the image it is given and crops to the resulting box (`predict` of Method B). The last
    decision is kept in ``last`` for the printout."""

    stochastic = False

    def __init__(self, model: nn.Module, device, aspect: float, size_div: int = 1):
        self.model, self.device, self.aspect, self.size, self.last = model, device, float(aspect), seg_size(size_div), None

    def __call__(self, rgb: np.ndarray, record=None) -> np.ndarray:
        post = segmenter_posts(self.model, [rgb], self.device, self.size, amp=str(self.device) == "cuda")[0]
        height, width = rgb.shape[:2]
        crop = (0, 0, width, height) if post.fallback else crop_box_pixels(post.box, self.aspect, width, height)
        self.last = {"crop": crop, "fallback": post.fallback, "area": post.area}
        x0, y0, x1, y1 = crop
        return np.ascontiguousarray(rgb[y0:y1, x0:x1])


BOXES_FILENAME = "boxes.json"
SEG_METRICS_FILENAME = "seg_metrics.json"


def seg_runs_root(data_root, seg_dir=None) -> Path:
    """The folder that holds the S-seg split folders: ``_work/runs/S-seg`` unless ``seg_dir`` is given."""
    return Path(seg_dir) if seg_dir is not None else runs_dir(data_root) / "S-seg"


def boxes_path(data_root, split_seed: int, seg_dir=None) -> Path:
    return split_dir(seg_runs_root(data_root, seg_dir), split_seed) / BOXES_FILENAME


def file_sha256(path: Path) -> str:
    return hashlib.sha256(read_file_bytes(path)).hexdigest()


def load_boxes(data_root, split_seed: int, seg_dir=None) -> dict:
    """``{filename: {"tight", "box", "fallback", ...}}`` of the split-``k`` segmenter, or a clear ExperimentError."""
    path = boxes_path(data_root, split_seed, seg_dir)
    if not path.is_file():
        raise ExperimentError(
            f"{path} does not exist: Method B (E4-B) crops by the boxes of the split-{split_seed} segmenter. "
            f"Run `run --exp S-seg --splits {split_seed}` first (it writes boxes.json after training)"
        )
    document = json.loads(read_file_bytes(path).decode("utf-8"))
    return document["images"]


def compute_boxes(model: nn.Module, data_root, records: list[Record], device, size, *, amp: bool = False, batch_size: int = 32) -> tuple[dict, dict]:
    """(boxes, fallback counts per domain) of the segmenter over every record (cached images): the document written to boxes.json."""
    boxes, counts = {}, {domain: {"n": 0, "fallback": 0} for domain in DOMAINS}
    for start in range(0, len(records), batch_size):
        chunk = records[start : start + batch_size]
        posts = segmenter_posts(model, [read_cached_rgb(data_root, r.filename) for r in chunk], device, size, amp=amp)
        for record, post in zip(chunk, posts):
            boxes[record.filename] = {
                "domain": record.domain, "fallback": post.fallback, "area": round(post.area, 5),
                "tight": None if post.tight is None else [round(v, 5) for v in post.tight],
                "box": [round(v, 5) for v in post.box],
            }
            counts[record.domain]["n"] += 1
            counts[record.domain]["fallback"] += int(post.fallback)
    return boxes, counts


def write_boxes(path: Path, split_seed: int, boxes: dict, counts: dict, exp_id: str) -> str:
    """Write boxes.json (one image per line) and return its sha256."""
    header = {"format": 1, "split": split_seed, "exp_id": exp_id, "box_recipe": BOX_RECIPE, "margin": SEG_MARGIN, "min_area": SEG_MIN_AREA,
              "threshold": SEG_THRESHOLD, "fallbacks": counts}
    head = ",\n".join(f" {json.dumps(k)}: {json.dumps(v)}" for k, v in header.items())
    body = ",\n".join(f"  {json.dumps(name)}: {json.dumps(entry)}" for name, entry in sorted(boxes.items()))
    write_text_atomic(path, "{\n" + head + ',\n "images": {\n' + body + "\n }\n}\n")
    return file_sha256(path)


def load_seg_run_model(run_dir, device: str = "cpu") -> tuple[nn.Module, dict]:
    """(segmenter in eval mode on ``device``, checkpoint dict) from <run_dir>/best.pt of an S-seg split folder."""
    path = Path(run_dir) / "best.pt"
    if not path.is_file():
        raise ExperimentError(
            f"{path} does not exist. Method B needs the segmenter checkpoint of its split to segment a new image; checkpoints "
            "are not synced back from Kaggle by default (*.pt is excluded): keep or restore _work/runs/S-seg/split_<k>/best.pt"
        )
    checkpoint = torch.load(fs_path(path), map_location="cpu", weights_only=True)
    model = build_segmenter(checkpoint["arch"], pretrained=False)
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(device).eval(), checkpoint

## Segmenter metrics on the annotated field images, and the leakage rule

The 30 annotated images split into **10 dev** (2 per class) and **20 eval** (4 per class). The rule of 06: development and
tuning may use the dev images only; each *final* segmenter looks at the eval images once, for reporting.
The code enforces that structurally:

* `run --exp S-seg` (and `smoke`) never read an annotation: they train, select the checkpoint on validation composites and
  white val masks, and write `boxes.json`. Their output holds validation numbers only.
* **`seg-eval --set dev`** measures the dev images (allowed any time; prints per-image results and writes the `dev` section of
  `seg_metrics.json`). Its figure is `seg_examples_dev.png` next to the run.
* **`seg-eval --set eval --final`** measures the eval images. It refuses to run without `--final`, and refuses a second run on
  the same segmenter (same boxes) unless `--again` is given, so the number is looked at once. It writes the `eval` section of
  `seg_metrics.json` and, for split 0, the report figure `_work/figures/seg_examples.png`. `summarize` only *reads* that section.

**Metrics** (per segmenter, then mean +- std over the 5 split-segmenters; the table is `table_segmenter`):

* **IoU / Dice** between the post-processed mask (the selected component) and the annotated polygon, both at 512 x 256, over
  all images of the set and again excluding the ones flagged `ambiguous`. A fallback has an empty mask, so it scores 0.
* **Salient-leaf hit rate**: the share of images whose crop box (the final box at the classifier's input aspect, from the
  chosen input mode; `rect` while gate G0 has not run) contains at least 50% of the annotated leaf area. It matters more than IoU,
  because Method B only uses the box. A fallback crop is the whole image and therefore a hit; fallbacks are counted separately.
* **Crop-box area**: the mean share of the image the crop box covers. A box that always covers the whole image (every
  fallback does) always "hits", so the hit rate is only meaningful next to this number and the fallback count.
* **IoU on the white val pseudo-masks** and on the validation composites (the checkpoint-selection data; a sanity check).

In [ ]:
#| export
SEG_EVAL_SETS = ("dev", "eval")
HIT_FRACTION = 0.5  # the crop box must contain at least this share of the annotated leaf area


def annotation_entries(data_root, which: str, annotations=None) -> list[dict]:
    """The annotated images of one set ("dev" or "eval"): ``{"filename", "cls", "set", "ambiguous", "polygon"}`` with the
    polygon as an array of fractional (x, y). ``annotations`` default to the embedded ``FIELD_ANNOTATIONS`` and are checked
    against the committed selection."""
    if which not in SEG_EVAL_SETS:
        raise ExperimentError(f"unknown annotation set {which!r}, expected one of {SEG_EVAL_SETS}")
    annotations = embedded_field_annotations() if annotations is None else annotations
    if annotations is None:
        raise ExperimentError("there are no annotations: annotations/field_salient_leaf.json is not embedded (see docs/instructions/annotation-guide.md)")
    selection = load_selection(data_root)
    validate_field_annotations(annotations, selection)
    return [
        {"filename": row["filename"], "cls": row["cls"], "set": row["set"], "ambiguous": bool(annotations[row["filename"]]["ambiguous"]),
         "polygon": np.asarray(annotations[row["filename"]]["polygon"], dtype=np.float64)}
        for row in selection if row["set"] == which
    ]


def rasterize_polygon(polygon, size: tuple[int, int]) -> np.ndarray:
    """Boolean mask [h, w] of a polygon given as fractional (x, y) vertices."""
    height, width = size
    canvas = np.zeros((height, width), np.uint8)
    cv2.fillPoly(canvas, [np.round(np.asarray(polygon) * (width, height)).astype(np.int32)], 1)
    return canvas.astype(bool)


def iou_dice(pred: np.ndarray, truth: np.ndarray) -> tuple[float, float]:
    inter = float((pred & truth).sum())
    union = float((pred | truth).sum())
    total = float(pred.sum() + truth.sum())
    return (inter / union if union else 1.0), (2 * inter / total if total else 1.0)


def box_fraction(crop_fractions, truth: np.ndarray) -> float:
    """Share of the ``truth`` mask's area that lies inside the box ``(x0, y0, x1, y1)`` (fractions of the image)."""
    height, width = truth.shape
    x0, y0 = int(math.floor(crop_fractions[0] * width + 1e-6)), int(math.floor(crop_fractions[1] * height + 1e-6))
    x1, y1 = int(math.ceil(crop_fractions[2] * width - 1e-6)), int(math.ceil(crop_fractions[3] * height - 1e-6))
    return float(truth[y0:y1, x0:x1].sum()) / max(1.0, float(truth.sum()))


def evaluate_annotations(model: nn.Module, data_root, entries: list[dict], device, size, aspect: float, *, amp: bool = False) -> tuple[list, list]:
    """(per-image results, SegPosts) of a segmenter on annotated images. Each result: filename, class, set, ambiguous, iou,
    dice, hit (crop box contains >= 50% of the leaf), fraction_in_box, fallback, area, crop (fractional box)."""
    rgbs = [read_cached_rgb(data_root, e["filename"]) for e in entries]
    posts = segmenter_posts(model, rgbs, device, size, amp=amp) if rgbs else []
    results = []
    for entry, rgb, post in zip(entries, rgbs, posts):
        height, width = rgb.shape[:2]
        truth = rasterize_polygon(entry["polygon"], post.mask.shape)
        crop = (0, 0, width, height) if post.fallback else crop_box_pixels(post.box, aspect, width, height)
        crop_fractions = (crop[0] / width, crop[1] / height, crop[2] / width, crop[3] / height)
        iou, dice = iou_dice(post.mask, truth)
        inside = box_fraction(crop_fractions, truth)
        results.append({
            "filename": entry["filename"], "cls": entry["cls"], "set": entry["set"], "ambiguous": entry["ambiguous"],
            "iou": iou, "dice": dice, "hit": inside >= HIT_FRACTION, "fraction_in_box": inside, "fallback": post.fallback,
            "area": post.area, "crop": [round(v, 5) for v in crop_fractions],
        })
    return results, posts


def summarize_annotation_results(results: list[dict]) -> dict:
    """Mean IoU, Dice, hit rate and crop-box area (share of the image) over all images and over the non-ambiguous ones,
    and the fallback count."""
    def block(rows):
        if not rows:
            return {"n": 0, "iou": None, "dice": None, "hit_rate": None, "box_area": None}
        return {"n": len(rows), "iou": float(np.mean([r["iou"] for r in rows])), "dice": float(np.mean([r["dice"] for r in rows])),
                "hit_rate": float(np.mean([r["hit"] for r in rows])),
                "box_area": float(np.mean([(r["crop"][2] - r["crop"][0]) * (r["crop"][3] - r["crop"][1]) for r in rows]))}
    return {"all": block(results), "excluding_ambiguous": block([r for r in results if not r["ambiguous"]]),
            "fallbacks": int(sum(r["fallback"] for r in results))}


def example_order(results: list[dict], n: int = 10, failures: int = 2) -> list[int]:
    """Indices of ``n`` results for the examples figure: the ``failures`` lowest-IoU images go last, the others are spread over
    the IoU range (best first). Deterministic."""
    order = sorted(range(len(results)), key=lambda i: (-results[i]["iou"], results[i]["filename"]))
    if len(order) <= n:
        return order
    worst, rest = order[len(order) - failures :], order[: len(order) - failures]
    picked = [rest[int(round(k))] for k in np.linspace(0, len(rest) - 1, n - failures)]
    return picked + worst


def show_photo(ax, fig, rgb: np.ndarray) -> None:
    """Draw a photo 1:1: it is downsampled once, with an area filter, to the pixels the axes occupy at FIGURE_DPI."""
    box = ax.get_position()
    tile = (max(1, round(box.width * fig.get_figwidth() * FIGURE_DPI)), max(1, round(box.height * fig.get_figheight() * FIGURE_DPI)))
    height, width = rgb.shape[:2]
    ax.imshow(cv2.resize(rgb, tile, interpolation=cv2.INTER_AREA), extent=(0, width, height, 0), interpolation="none")


def save_seg_examples(data_root, entries: list[dict], results: list[dict], posts: list[SegPost], path: Path, *, n: int = 10, failures: int = 2) -> Path:
    """`seg_examples.png`: ``n`` annotated field images (``failures`` of them the worst), each with the predicted mask (blue),
    the annotation (yellow outline) and the crop box (vermilion). Needs ``results`` and ``posts`` of the same images."""
    plt = pyplot()
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    picks = example_order(results, n, failures)
    columns = 5
    rows = math.ceil(len(picks) / columns)
    path.parent.mkdir(parents=True, exist_ok=True)
    worst = set(picks[-failures:]) if len(results) > n else set()
    with plt.rc_context({"font.size": 6}):
        fig = plt.figure(figsize=(5.8, 2.42 * rows + 0.3))
        grid = fig.add_gridspec(rows, columns, left=0.005, right=0.995, top=0.995, bottom=0.075, wspace=0.05, hspace=0.2)
        for slot, index in enumerate(picks):
            result, post = results[index], posts[index]
            ax = fig.add_subplot(grid[slot // columns, slot % columns])
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            rgb = read_cached_rgb(data_root, result["filename"])
            height, width = rgb.shape[:2]
            show_photo(ax, fig, rgb)
            scale = np.array([width / post.mask.shape[1], height / post.mask.shape[0]])
            overlay = np.zeros((*post.mask.shape, 4), np.float32)
            overlay[post.mask] = (*[int(OKABE_ITO["sky"][i : i + 2], 16) / 255 for i in (1, 3, 5)], 0.5)
            ax.imshow(overlay, extent=(0, width, height, 0), interpolation="nearest")
            contours, _ = cv2.findContours(post.mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            for contour in contours:
                points = (contour[:, 0, :] + 0.5) * scale
                closed = np.vstack([points, points[:1]])
                ax.plot(closed[:, 0], closed[:, 1], color=OKABE_ITO["blue"], linewidth=0.7)
            polygon = entries[index]["polygon"] * (width, height)
            closed = np.vstack([polygon, polygon[:1]])
            ax.plot(closed[:, 0], closed[:, 1], color="black", linewidth=1.5, solid_joinstyle="round")
            ax.plot(closed[:, 0], closed[:, 1], color=OKABE_ITO["yellow"], linewidth=0.7, solid_joinstyle="round")
            if not result["fallback"]:  # a fallback crop is the whole frame: drawing it would only outline the photo
                x0, y0, x1, y1 = result["crop"]
                ax.plot([x0 * width, x1 * width, x1 * width, x0 * width, x0 * width], [y0 * height, y0 * height, y1 * height, y1 * height, y0 * height],
                        color=OKABE_ITO["vermilion"], linewidth=1.0, linestyle=(0, (3, 1.5)))
            ax.set_xlim(0, width)
            ax.set_ylim(height, 0)
            verdict = "fallback" if result["fallback"] else "box hit" if result["hit"] else "box miss"
            ax.set_xlabel(f"{result['filename']}\nIoU {result['iou']:.2f}, {verdict}", fontsize=5.5, labelpad=1.5,
                          color=OKABE_ITO["vermilion"] if index in worst else "black")
        handles = [Patch(fc=OKABE_ITO["sky"], ec=OKABE_ITO["blue"], alpha=0.7, label="predicted mask"),
                   Line2D([0], [0], color=OKABE_ITO["yellow"], lw=1.2, label="annotation"),
                   Line2D([0], [0], color=OKABE_ITO["vermilion"], lw=1.2, linestyle=(0, (3, 1.5)), label="crop box"),
                   Line2D([], [], linestyle="none", label="fallback: no leaf found, whole image used")]
        fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=6, bbox_to_anchor=(0.5, 0.0),
                   handlelength=1.6, columnspacing=1.2)
        fig.savefig(path, dpi=FIGURE_DPI)
        plt.close(fig)
    return path



def read_seg_metrics(run_dir: Path) -> dict:
    path = Path(run_dir) / SEG_METRICS_FILENAME
    return json.loads(read_file_bytes(path).decode("utf-8")) if path.is_file() else {}


def run_seg_eval(data_root, which: str, splits, *, final: bool = False, again: bool = False, seg_dir=None, device: str = "cpu",
                 input_mode: str | None = None, annotations=None, figure: Path | None = None, out=print) -> dict:
    """`seg-eval`: segmenter metrics on the annotated field images of one set (see the section text for the rule).
    ``which="eval"`` needs ``final=True`` and, once measured, ``again=True`` to measure the same segmenter again.
    Returns {split: summary}."""
    data_root = Path(data_root)
    if which not in SEG_EVAL_SETS:
        raise ExperimentError(f"unknown annotation set {which!r}, expected one of {SEG_EVAL_SETS}")
    if which == "eval" and not final:
        raise ExperimentError(
            "the 20 eval annotations are looked at once per final segmenter, for reporting only (06, leakage rule): "
            "do not use them while developing (use --set dev). If this IS the final segmenter, add --final"
        )
    choices = load_choices(data_root)
    mode = input_mode or choices.raw("input_mode")
    if mode is None:
        mode = "rect"
        out("seg-eval: choices.json has no input_mode yet; the hit rate uses the rect crop aspect (448x224)")
    entries = annotation_entries(data_root, which, annotations)
    aspect = classifier_aspect(mode)
    summaries, shown_runs = {}, {}
    for split_seed in splits:
        run_dir = split_dir(seg_runs_root(data_root, seg_dir), split_seed)
        metrics = read_metrics(run_dir)
        if metrics is None:
            raise ExperimentError(f"{run_dir / 'metrics.json'} is missing: train the segmenter first (`run --exp S-seg --splits {split_seed}`)")
        record = read_seg_metrics(run_dir)
        previous = record.get(which)
        if which == "eval" and previous and previous.get("boxes_sha256") == metrics.get("boxes_sha256") and not again:
            raise ExperimentError(
                f"split {split_seed}: the eval annotations were already measured for this segmenter ({run_dir / SEG_METRICS_FILENAME}). "
                "They are looked at once; pass --again only if the segmenter changed or the first run was a mistake"
            )
        model, checkpoint = load_seg_run_model(run_dir, device)
        size = seg_size(checkpoint.get("size_div", 1))
        results, posts = evaluate_annotations(model, data_root, entries, device, size, aspect, amp=device == "cuda")
        summary = summarize_annotation_results(results)
        record[which] = {
            "set": which, "input_mode": mode, "hit_fraction": HIT_FRACTION, "config_fingerprint": metrics.get("config_fingerprint"),
            "boxes_sha256": metrics.get("boxes_sha256"), "best_epoch": metrics.get("best_epoch"), "summary": summary, "images": results,
        }
        write_json(run_dir / SEG_METRICS_FILENAME, record, indent=1)
        summaries[split_seed] = summary
        shown_runs[split_seed] = (run_dir, results, posts)
        a, e = summary["all"], summary["excluding_ambiguous"]
        out(f"seg-eval {which} split {split_seed}: IoU {a['iou']:.3f} Dice {a['dice']:.3f} hit {a['hit_rate']:.2f} (n={a['n']}); "
            f"excluding ambiguous IoU {e['iou']:.3f} Dice {e['dice']:.3f} hit {e['hit_rate']:.2f} (n={e['n']}); "
            f"crop box {100 * a['box_area']:.0f}% of the image; fallbacks {summary['fallbacks']}")
        if which == "dev":
            for r in results:
                out(f"    {r['filename']:<12}{r['cls']:<14} IoU {r['iou']:.3f} Dice {r['dice']:.3f}  box has {100 * r['fraction_in_box']:.0f}% of the leaf"
                    f"{'  FALLBACK' if r['fallback'] else ''}{'  (ambiguous)' if r['ambiguous'] else ''}")
    shown = shown_runs.get(0) if which == "eval" else next(iter(shown_runs.values()), None)
    if shown is not None:
        run_dir, results, posts = shown
        target = figure if figure is not None else figures_dir(data_root) / "seg_examples.png" if which == "eval" else run_dir / "seg_examples_dev.png"
        out(f"seg-eval: figure {save_seg_examples(data_root, entries, results, posts, Path(target))}")
    return summaries


def cmd_seg_eval(args: "argparse.Namespace") -> int:
    try:
        run_seg_eval(Path(args.data_root), args.set, args.splits, final=args.final, again=args.again, seg_dir=args.seg_dir,
                     device=resolve_device(args.device), figure=args.figure)
    except (ExperimentError, DatasetError) as exc:
        print(f"seg-eval: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

## `run --exp S-seg` and the Method B experiment (T6.2, T6.3)

`run_seg_split` is the segmenter's counterpart of `run_split` and uses the same machinery: the run folder
`_work/runs/S-seg/split_<k>/` is written atomically with `metrics.json` last; a split is skipped when its `metrics.json` is
current (same split sha256 and **config fingerprint**) unless `--force`; `--fast` and the dev overrides go to
`_work/runs/scratch/`.

| File | Content |
|---|---|
| `config.json`, `env.json`, `history.csv` | as for classifiers; the history has one row per epoch with the validation IoUs |
| `best.pt` | `{state_dict, arch, size_div, ...}` of the best epoch (needed to segment a new image: `predict` for E4-B, `seg-eval`) |
| `boxes.json` | the crop boxes of **all** cached images (T6.3), see above |
| `seg_metrics.json` | validation IoU of the chosen epoch (composites, white), fallback counts per domain; `seg-eval` adds the `dev` / `eval` sections |
| `metrics.json` | split sha256, fingerprint, timing, the numbers above, sha256 of `boxes.json`. Written last |

The **fingerprint** of an S-seg run covers the registry row plus everything that defines the training data and recipe: the
segmenter constants (`SEG_RECIPE`), the composite and mask recipes (`COMPOSITE_RECIPE`) and the reject list
(`load_white_rejects`; a rejected leaf is no longer a target). Classifier rows add nothing, so no existing fingerprint moves;
E4-B adds only `BOX_RECIPE` (the post-processing and jitter constants), and its metrics record the sha256 of the `boxes.json`
it used, so a retrained segmenter marks the E4-B runs of that split as stale.

In [ ]:
#| export
def seg_fingerprint_extra(cfg: ExperimentConfig, data_root=None) -> dict:
    """What the segmenter row and the Method B row add to their config fingerprint; ``{}`` for every other row, so their
    fingerprints are exactly what they were before phase 06."""
    if cfg.kind == "segmenter":
        rejects = sorted(load_white_rejects(data_root)) if data_root is not None else []
        digest = hashlib.sha1("\n".join(rejects).encode()).hexdigest() if rejects else ""
        return {"segmenter": {"recipe": SEG_RECIPE, "composites": COMPOSITE_RECIPE, "rejects": digest}}
    if cfg.seg_crop:
        return {"seg_crop": {"recipe": BOX_RECIPE}}
    return {}


def run_seg_split(resolved: "ResolvedExperiment", split_seed: int, *, data_root, out_dir: Path, options: "RunOptions", log=print) -> dict:
    """Train the split-``k`` segmenter, pick the best epoch on validation, write the run folder and the boxes of every cached
    image (see the section text). Returns {"status": "skipped" | "done", "metrics": dict, "out_dir": Path}."""
    data_root, out_dir = Path(data_root), Path(out_dir)
    cfg = resolved.config
    split = load_split(data_root, split_seed)
    fingerprint = config_fingerprint(cfg, {"seed_offset": options.seed_offset, "limit": options.limit, **seg_fingerprint_extra(cfg, data_root)})
    state, reason = result_state(out_dir, split["sha256"], fingerprint)
    if state == "fresh" and not options.force:
        log(f"[{resolved.requested} split {split_seed}] skipped: {out_dir / 'metrics.json'} exists and is current (--force to retrain)")
        return {"status": "skipped", "metrics": read_metrics(out_dir), "out_dir": out_dir}
    if state == "stale":
        log(f"[{resolved.requested} split {split_seed}] existing result is stale: {reason}; retraining")
    (out_dir / "metrics.json").unlink(missing_ok=True)  # a half-finished retrain must not look complete

    started = time.time()
    seed = split_seed + options.seed_offset
    records = load_index(data_root)
    rejects = load_white_rejects(data_root, records)
    whites = white_mask_records(data_root, records, split["train"], rejects)
    if options.limit:
        whites = subsample_records(whites, options.limit)
    build_white_masks(data_root, whites)
    size = seg_size(cfg.size_div)
    n_samples = options.limit or SEG_SAMPLES_PER_EPOCH
    pool = CompositePool(data_root, split_seed, records, rejects=rejects)
    n_composites = round(SEG_COMPOSITE_SHARE * n_samples) if pool.leaves else 0
    if n_composites:
        check_cache_complete(data_root, [*pool.leaves, *(b for group in pool.backgrounds.values() for b in group)])
    composites = CompositeSource(data_root, split_seed, n_composites, seed, pool=pool) if n_composites else None
    train_ds = SegmenterDataset(data_root, whites, composites, size=size, n_samples=n_samples, seed=seed)
    val_composites, val_white = build_seg_val_sets(data_root, split_seed, records, size, limit=options.limit)
    workers = default_num_workers() if options.num_workers is None else options.num_workers
    log(f"[{resolved.requested} split {split_seed}] {cfg.exp_id}: {cfg.backbone}  input {size[0]}x{size[1]}  lr {cfg.lr:g}  "
        f"train {len(train_ds)}/epoch ({train_ds.n_composites} composites + {len(train_ds) - train_ds.n_composites} white), "
        f"val {len(val_composites)} composites + {len(val_white)} white  device {options.device}, batch {cfg.batch_size}, workers {usable_workers(workers)}")
    write_json(out_dir / "config.json", {
        "format": RUN_FORMAT, "requested_exp_id": resolved.requested, "experiment": asdict(cfg), "split": split_seed, "seed": seed,
        "seed_offset": options.seed_offset, "limit": options.limit, "overrides": dict(options.overrides),
        "weights": SEG_ARCHITECTURES[cfg.backbone] if cfg.pretrained else None, "input_size": list(size),
        "segmenter": {"constants": {k: list(v) if isinstance(v, tuple) else v for k, v in SEG_CONSTANTS.items()}, "recipe": SEG_RECIPE,
                      "composite_recipe": COMPOSITE_RECIPE, "rejects": len(rejects), "samples_per_epoch": n_samples},
        "config_fingerprint": fingerprint,
    })
    write_json(out_dir / "env.json", environment_record(options.device))

    seed_everything(seed)
    model = build_segmenter(cfg.backbone, pretrained=cfg.pretrained, data_root=options.weights_root).to(options.device)
    outcome = train_segmenter(cfg, model, train_ds, val_composites, val_white, options.device, seed=seed, num_workers=workers,
                              mp_context=options.mp_context, log=log)
    model.load_state_dict(outcome.best_state)
    amp = options.device == "cuda"
    composite_iou, white_iou = seg_val_iou(model, val_composites, options.device, amp=amp), seg_val_iou(model, val_white, options.device, amp=amp)
    boxes, counts = compute_boxes(model, data_root, records, options.device, size, amp=amp)
    for domain, c in counts.items():
        c["rate"] = c["fallback"] / c["n"] if c["n"] else None
    boxes_sha = write_boxes(out_dir / BOXES_FILENAME, split_seed, boxes, counts, cfg.exp_id)
    write_text_atomic(out_dir / "history.csv", csv_text(SEG_HISTORY_COLUMNS, [
        [f"{row[c]:.6g}" if isinstance(row[c], float) else "" if row[c] is None else row[c] for c in SEG_HISTORY_COLUMNS] for row in outcome.history]))
    save_torch_atomic({
        "format": RUN_FORMAT, "state_dict": outcome.best_state, "arch": cfg.backbone, "size_div": cfg.size_div, "exp_id": cfg.exp_id,
        "split": split_seed, "epoch": outcome.best_epoch, "val_score": outcome.best_score, "config": asdict(cfg),
    }, out_dir / "best.pt")
    write_json(out_dir / SEG_METRICS_FILENAME, {
        "format": 1, "split": split_seed, "best_epoch": outcome.best_epoch, "boxes_sha256": boxes_sha,
        "val": {"composite_iou": composite_iou, "white_iou": white_iou, "score": outcome.best_score,
                "n_composites": len(val_composites), "n_white": len(val_white)},
        "fallbacks": counts,
    }, indent=1)
    epoch_seconds = [row["seconds"] for row in outcome.history]
    metrics = {
        "format": RUN_FORMAT, "exp_id": cfg.exp_id, "requested_exp_id": resolved.requested, "kind": "segmenter", "split": split_seed,
        "seed": seed, "split_sha256": split["sha256"], "config_fingerprint": fingerprint,
        "wall_time_s": round(time.time() - started, 2), "train_time_s": round(outcome.seconds, 2),
        "seconds_per_epoch": round(float(np.mean(epoch_seconds)), 3) if epoch_seconds else None,
        "device": options.device, "arch": cfg.backbone, "lr": cfg.lr, "parameters": count_parameters(model),
        "n_train_per_epoch": len(train_ds), "n_composites_per_epoch": train_ds.n_composites, "n_val_composites": len(val_composites),
        "n_val_white": len(val_white), "epochs_run": len(outcome.history), "best_epoch": outcome.best_epoch,
        "best_val_score": outcome.best_score, "val_iou_composite": composite_iou, "val_iou_white": white_iou,
        "fallbacks": counts, "boxes_sha256": boxes_sha, "n_boxes": len(boxes), "fast": options.limit is not None,
        "overrides": dict(options.overrides),
    }
    write_json(out_dir / "metrics.json", metrics, indent=1)  # last: its presence marks the run as complete
    fmt = lambda v: "n/a" if v is None else f"{v:.3f}"  # noqa: E731
    log(f"[{resolved.requested} split {split_seed}] done in {metrics['wall_time_s']:.0f}s  best epoch {outcome.best_epoch}  val IoU composites "
        f"{fmt(composite_iou)} white {fmt(white_iou)}; boxes for {len(boxes)} images, fallbacks white {counts['white']['fallback']}/{counts['white']['n']} "
        f"field {counts['field']['fallback']}/{counts['field']['n']}")
    return {"status": "done", "metrics": metrics, "out_dir": out_dir}